# Lab03B — assimetria, setores e risco em todo o período
**Além do Beta · CMP609 · 23/10/2026**

Este caderno reúne as quatro prioridades do artigo: assimetria por ativo, heterogeneidade setorial, dinâmica temporal e extensões condicionadas à viabilidade. Os Labs X/Y/03 salvos pelo autor foram conferidos; não é preciso repeti-los. Executamos um lote completo de síntese e risco usando seus arquivos autenticados.

São 80 ativos fixos em 2016–2025, 117 fixos em 2021–2025 e os mesmos 80 no período comum, dez setores, sem carteira ou pesos por empresa. **Modelos usam retornos diários observados.** Dados incompletamente homologados continuam condicionando a interpretação; este notebook não resolve o gate global.

## 1. O que o CAPM acrescenta — e o que foi calculado
O material de aula fornecido em 23/10 discute CAPM, prêmio de risco, beta, alfa, diversificação, alavancagem e APT. O CAPM relaciona retorno esperado e exposição ao mercado:

$$E(R_i)-R_f=\beta_i[E(R_m)-R_f].$$

A equação de equilíbrio não converte retorno observado em previsão validada. Nosso modelo de mercado é $r_i=a_i+\beta_i m+e_i$, sobre log-retornos observados. O intercepto deste modelo **não é alfa de Jensen**; não foi validada uma série livre de risco contemporânea para um novo teste em retornos excedentes. Não calculamos Sharpe/Treynor nem carteiras.

Literatura: Sharpe (1964), Ang/Chen/Xing (2006) e Levi/Welch (2020). A última referência contrasta resultados ex ante e ex post: diferenças estimadas não demonstram automaticamente superioridade preditiva do downside beta. Métodos de risco de cauda são complemento explícito, não conteúdo atribuído integralmente à aula.

In [ ]:
from pathlib import Path
import os,sys,io,json,hashlib,zipfile,tempfile,datetime,subprocess,platform
import numpy as np
import pandas as pd
from IPython.display import display,Image
IN_COLAB='COLAB_RELEASE_TAG' in os.environ
FOLDER_ID='1kAPlpmnedTOdDx6wunZ7nl0YEqf3tAR9'
RUNTIME_ID='1UsIkq60ZNQqWVznfqbnbOsSRFpciZRcr'
MANIFEST_ID='1ZPrV0-pu3GcTNS6kew1giaIJr58c0sOU'
EXPECTED_MANIFEST_SHA='7e84b52a33ed337f25e624f27d1210c96ef0d4a9cd595150b7580a0b839aeb92'
if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload,MediaFileUpload
    credentials,_=google.auth.default()
    drive_service=build('drive','v3',credentials=credentials,cache_discovery=False)
    def download_bytes(file_id):
        stream=io.BytesIO();request=MediaIoBaseDownload(stream,drive_service.files().get_media(fileId=file_id));done=False
        while not done:_,done=request.next_chunk()
        return stream.getvalue()
    manifest_bytes=download_bytes(MANIFEST_ID);package_bytes=download_bytes(RUNTIME_ID)
    CACHE=Path('/content/alem_beta_y_cache');CACHE.mkdir(exist_ok=True)
else:
    PACKAGE_DIR=Path(os.environ['LAB03B_PACKAGE_DIR'])
    manifest_bytes=(PACKAGE_DIR/'lab03b_runtime_manifest.json').read_bytes();package_bytes=(PACKAGE_DIR/'lab03b_runtime_v1.zip').read_bytes()
assert hashlib.sha256(manifest_bytes).hexdigest()==EXPECTED_MANIFEST_SHA
runtime=json.loads(manifest_bytes);assert hashlib.sha256(package_bytes).hexdigest()==runtime['package_sha256']
REPO=Path(tempfile.mkdtemp(prefix='alem_beta_lab03b_'))
with zipfile.ZipFile(io.BytesIO(package_bytes)) as z:
    assert len(z.namelist())==len(set(z.namelist())) and set(z.namelist())==set(runtime['files'])
    for name,item in runtime['files'].items():
        p=Path(name);assert not p.is_absolute() and '..' not in p.parts
        raw=z.read(name);assert len(raw)==item['bytes'] and hashlib.sha256(raw).hexdigest()==item['sha256']
        target=REPO/p;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
lock=json.loads((REPO/'data/lab03b/inputs_manifest.json').read_text());archives={}
for key in ['x','y','lab03']:
    if IN_COLAB:
        path=CACHE/(lock[key+'_sha256']+'.zip')
        if not path.exists():
            raw=download_bytes(lock[key+'_drive_id']);assert hashlib.sha256(raw).hexdigest()==lock[key+'_sha256'];path.write_bytes(raw)
    else:path=Path(os.environ['LAB03B_'+key.upper()+'_ARCHIVE'])
    assert hashlib.sha256(path.read_bytes()).hexdigest()==lock[key+'_sha256'];archives[key]=path
X_ARCHIVE,Y_ARCHIVE,LAB03_ARCHIVE=[archives[k] for k in ['x','y','lab03']]
sys.path[:0]=[str(REPO/'src'),str(REPO)];ENV=dict(os.environ,PYTHONPATH=str(REPO/'src')+os.pathsep+str(REPO))
OUT=Path(os.environ.get('LAB03B_OUTPUT',str(REPO/'resultados_lab03b')))
print('Runtime e três ZIPs autenticados; sem download de cotações. Ambiente:', 'Colab' if IN_COLAB else 'preparação local')
print('Saídas:',OUT)

## 2. Proveniência e plano explícito
O PDF de aula tem 58 páginas; nome, data de impressão, SHA256 e cópia preservada constam de `material_aula.json`. Os métodos são adaptados com verificação de unidades, frequência e variáveis; não copiamos os exemplos como se fossem uma validação financeira.

O plano de síntese foi registrado antes desta execução, mas os resultados X/Y/03 já eram conhecidos. Não é pré-registro independente. A versão principal é N/log/HAC5; pisos e famílias 197/474 não mudam. Ausências completas (sete na década), Light e classes permanecem. O mesmo ativo no comum usa os mesmos resultados dos 117.

In [ ]:
display(pd.DataFrame([json.loads((REPO/'data/lab03b/material_aula.json').read_text())]))
print(json.dumps(json.loads((REPO/'data/lab03b/plano_v1.json').read_text()),indent=2,ensure_ascii=False))

## 3. Verificações científicas
Doze testes verificam exposição perfeita/negativa, decomposição da variância, ortogonalidade residual, pares ausentes, calendário, piso de 200 observações, cauda fracionária, empates, perdas com sinal e distinção entre mercado negativo e retorno do ativo abaixo de zero. Setor com coeficientes constantes/ausentes não recebe variância explicada artificial.

In [ ]:
import unittest
suite=unittest.TestLoader().discover(str(REPO/'tests'),pattern='test_lab03b.py')
test_result=unittest.TextTestRunner(verbosity=1).run(suite)
assert test_result.wasSuccessful() and test_result.testsRun==12
print('12 testes Lab03B aprovados.')

## 4. Lote completo: risco e setores
Calculamos 1.385 casos de risco para as cinco variantes X e 1.785 casos anuais N. Não anualizamos automaticamente o desvio padrão. A decomposição usa o mesmo denominador $n-1$:

$$s_i^2=\widehat\beta_i^2s_m^2+s_e^2,\qquad R^2=\widehat\beta_i^2s_m^2/s_i^2.$$

A parcela associada ao mercado é uma identidade descritiva do ajuste OLS, não decomposição causal de risco nem prova de resíduos economicamente idiossincráticos. O desvio de perdas é $\sqrt{n^{-1}\sum\min(r_i,0)^2}$, distinto de condicionar ao mercado em queda.

Para VaR/ES usamos perdas simples diárias $L=1-\exp(r)$: quantil empírico inverso a 95% e média da massa superior de 5%, com ponderação fracionária na fronteira. Ganhos não são truncados. ES anual de estimação pode apoiar-se em cerca de 12 observações de cauda: é descrição histórica, não previsão ou validação de capital regulatório.

In [ ]:
process=subprocess.Popen([sys.executable,str(REPO/'scripts/run_lab03b.py'),'--x-archive',str(X_ARCHIVE),'--y-archive',str(Y_ARCHIVE),'--lab03-archive',str(LAB03_ARCHIVE),'--output',str(OUT)],env=ENV,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:print(line,end='',flush=True)
assert process.wait()==0
metrics=json.loads((OUT/'metricas.json').read_text());assert metrics['risk_full_rows']==1385 and metrics['risk_annual_rows']==1785
assert metrics['bootstrap_y_reproduced_rows']==831 and not metrics['CAPM_pricing_tested']
display(pd.DataFrame([metrics]))

## 5. Evidência por ativo antes de generalizar
Veja as três janelas e as contagens estáticas já validadas. Δβ<0 significa maior exposição estimada nas quedas, não maior volatilidade necessariamente. Contagem de sinais não é teste de prevalência; ativos e classes podem compartilhar fatores e emissores.

O Lab03 já calculou as 474 hipóteses temporais por configuração. Nenhuma rejeição primária após Holm não prova equivalência, estabilidade ou ausência de dependência. Não há teste de previsão futura ou de remuneração esperada do risco.

In [ ]:
summary=pd.read_csv(OUT/'summary.csv');display(summary)
betas=pd.read_csv(OUT/'asset_betas.csv')
display(betas[betas.delta_p_holm_197.lt(.05)][['scenario','ticker','delta','delta_p_holm_197']])

## 6. Incerteza setorial e tamanhos pequenos
Resumimos a distribuição dos coeficientes dos ativos por mediana/IQR e contraste mediana do setor menos mediana dos demais. Não estimamos retorno de carteira setorial nem atribuímos pesos a empresas. **Setor com n=1 descreve aquele ativo; não permite generalização setorial.** A classificação é o cadastro B3 datado, não composição histórica reconstruída.

Reutilizamos os mesmos 5.994 sorteios Y: blocos de 5/10/20 pregões, 999 réplicas por recorte original, sorteio diário síncrono entre todos os ativos. Os 831 erros padrão Y são reproduzidos; o comum usa os mesmos 80 sorteados nos 117. Não se descarta um ativo para completar uma réplica.

ICs percentis são **marginais e exploratórios**, condicionados à aproximação estacionária; não são uma nova família confirmatória Holm. η² resume o agrupamento das estimativas pontuais por setor: não é percentual de risco explicado ou efeito estrutural, e também contém erro de estimação.

In [ ]:
sectors=pd.read_csv(OUT/'sector_point.csv');display(sectors)
display(pd.read_csv(OUT/'sector_bootstrap.csv').query('block_length == 10'))
display(pd.read_csv(OUT/'sector_eta.csv'))

## 7. Seis gráficos completos, de 2016 a 2025
A trajetória anual mostra todos os dez anos e dez setores. Os mapas móveis incluem todos os 80/117 ativos, com o aquecimento inicial de 252 pregões em cinza; não inventamos janelas móveis em 2016/2021 antes de existir histórico suficiente. A ausência inicial do mapa não é ausência da análise anual.

As faixas anuais são dispersão entre ativos, não ICs. Os intervalos do contraste setorial são explicitamente marginais. Eixos comparáveis usam escalas comuns; nenhum extremo foi cortado ou winsorizado. Os pares, regressões e medidas de risco continuam diários. A unidade nunca passa a ser empresa ou carteira.

In [ ]:
subprocess.run([sys.executable,str(REPO/'scripts/plot_lab03b.py'),'--output',str(OUT),'--lab03-archive',str(LAB03_ARCHIVE)],check=True,env=ENV)
for name in ['lab03b_betas','lab03b_setores_anuais','lab03b_mapa_movel','lab03b_contrastes_setoriais','lab03b_risco','lab03b_risco_anual']:
    display(Image(filename=str(OUT/'figures'/(name+'.png')),width=1200))

## 8. Conferência independente
Outro cálculo, sem importar módulos de produção, reconstrói os 3.170 casos de risco por regressões/momentos e integra a distribuição empírica de perdas. Reproduz os 5.994 sorteios e os 90 intervalos setoriais/nove intervalos η², além da igualdade exata dos mesmos ativos no período comum. Aguarde a conclusão desta célula.

In [ ]:
result=subprocess.run([sys.executable,str(REPO/'scripts/validate_lab03b_outputs.py'),'--x-archive',str(X_ARCHIVE),'--y-archive',str(Y_ARCHIVE),'--output',str(OUT)],env=ENV,capture_output=True,text=True)
assert result.returncode==0,result.stderr
independent=json.loads(result.stdout);assert independent['risk_rows_checked']==3170 and independent['bootstrap_draws_independently_reproduced']==5994
display(pd.DataFrame([independent]))

## 9. Leitura hierárquica para o artigo
1. Assimetria: estimativas e incerteza por ativo nas amostras fixas.
2. Setores: padrões descritivos e heterogeneidade dentro/entre grupos, com dependência e n pequeno explícitos.
3. Tempo: anos/janelas e testes condicionais já conferidos; não rejeição não é equivalência.
4. Extensões: HAC/bootstrap examinados; indicadores e covariância sequencial continuam opcionais, sem alegação de execução.

O artigo de entrega é organizado por essas perguntas. O dossiê conserva a auditoria. A identidade global de todos os ajustes, as pendências de dados e a elegibilidade de um periódico específico para o PPGC ainda exigem confirmação; nenhum aceite é prometido. A contribuição não deve ser julgada pelo número de métricas ou páginas.

In [ ]:
risk=pd.read_csv(OUT/'risk_full.csv');checks=pd.read_csv(OUT/'bootstrap_y_check.csv')
assert risk.variance_identity_error.max()<1e-12 and len(checks)==831
print('Modelo de mercado e risco verificados; CAPM em retornos excedentes/alfa de Jensen não estimados.')
print('Global homologado:',metrics['global_homologated'],'; extensão por indicadores/sequencial não executada.')

## 10. Salvar a execução
O ZIP final contém todas as tabelas, seis PNGs/PDFs, manifesto e registro de ambiente. O runtime reutiliza X/Y/03 sem duplicar as matrizes. Salve o notebook com as saídas e avise após a conclusão. Preparação local e Colab efetivo são conferidos separadamente; a execução não homologa a qualidade econômica global dos dados.

In [ ]:
record=dict(stage='Lab03B',environment='Colab' if IN_COLAB else 'local preparation',executed_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(),python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,tests_run=test_result.testsRun,tests_successful=test_result.wasSuccessful(),independent_check=independent,input_x_sha256=lock['x_sha256'],input_y_sha256=lock['y_sha256'],input_lab03_sha256=lock['lab03_sha256'],CAPM_pricing_tested=False,Jensen_alpha_estimated=False,global_homologated=0)
(OUT/'registro_execucao.json').write_text(json.dumps(record,indent=2)+'\n')
hashes={str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.rglob('*')) if p.is_file() and p.name!='manifesto_exports.json'}
(OUT/'manifesto_exports.json').write_text(json.dumps(hashes,indent=2)+'\n');archive=OUT.with_suffix('.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file():z.write(p,str(p.relative_to(OUT)))
print('ZIP:',archive,'SHA256:',hashlib.sha256(archive.read_bytes()).hexdigest())
if IN_COLAB:
    name='resultados_lab03b_'+datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.zip'
    uploaded=drive_service.files().create(body={'name':name,'parents':[FOLDER_ID]},media_body=MediaFileUpload(str(archive),mimetype='application/zip',resumable=True),fields='id,webViewLink').execute()
    print('Resultados salvos no Drive:',uploaded.get('webViewLink','https://drive.google.com/file/d/'+uploaded['id']+'/view'))
else:print('Preparação local; Colab próprio ainda requer conferência.')